# Proceso de automatizacion contable

In [1]:
# importar librerias a usar
import pandas as pd
from clase_reportes import ReportClass
import numpy as np
import tkinter as tk
from tkinter import filedialog
import json
import re
import os
rc = ReportClass()


In [ ]:
ruta = rc.validar_ruta()
ruta_contabilidad = ruta / 'data' / 'contabilidad'
df_base = rc.consolidar_carpeta(ruta_carpeta=ruta_contabilidad / 'odoo' )

df_base = df_base.rename(columns={'Cuenta': 'Cuenta Origen'})
df_base['Cuenta'] = df_base['Cuenta Origen'].str.split(' ', regex=True).str[0]
df_base['Nombre cuenta'] = df_base['Cuenta Origen'].str.split(' ', regex=True).str[1:].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
df_base['N1'] = df_base['Cuenta Origen'].astype(str).str[0]
df_base['N2'] = df_base['Cuenta Origen'].astype(str).str[:2]
df_base['N3'] = df_base['Cuenta Origen'].astype(str).str[:4]

# Define la columna nivel
df_base['Nivel']  =np.where(df_base['N2']=='41', 'Ingreso Operativo',
        np.where(df_base['N2']=='42', 'Otros ingresos',
                np.where(df_base['N2']=='52', 'Gastos operacionales',
                        np.where(df_base['N2']=='53', 'Gastos No Operacionales',
                                    np.where(df_base['N2']=='61', 'Costo directo de ventas', 
                                            'Revisar'
                                    )
                        )
                )
        )
)

df_niveles =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='niveles')
df_detalle =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='detalle')
df_concepto =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='concepto')
df_cc =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='CC')
df_base['N3'] = df_base['N3'].astype(int)
df_base['N3'] = df_base['N3'].astype(int)
df_base['Cuenta'] = df_base['Cuenta'].astype(int)

df_base_merge = df_base.merge(df_niveles, left_on='N3', right_on='cuenta', how='left').drop(columns='cuenta')

def extraer_clave(diccionario_str):
    if pd.isna(diccionario_str):
        return None
    try:
        diccionario = json.loads(diccionario_str)
        return list(diccionario.keys())[0]
    except Exception:
        return None

df_base_merge = df_base_merge.rename(columns={'Distribución analítica': 'Distribución analítica ori'})

df_base_merge['Distribución analítica'] = df_base_merge['Distribución analítica ori'].apply(extraer_clave)


# Ajustes manuales de asignación de centro de costo y concepto
df_base_merge['N1'] = df_base_merge['N1'].astype(str)
df_base_merge['N2'] = df_base_merge['N2'].astype(str)
df_base_merge['N3'] = df_base_merge['N3'].astype(str)

df_base_merge.loc[
    (df_base_merge['N3'] == '4135'),
    'Distribución analítica', 
] = '6'



df_base_merge.loc[
    (df_base_merge['N1'] == '6') & (df_base_merge['Distribución analítica ori'].isna()),
    'Distribución analítica'
] =  '6'


df_base_merge.loc[
    (df_base_merge['N2'] == '42') & (df_base_merge['Distribución analítica ori'].isna()),
    'Distribución analítica'
] = '6'



df_base_merge.loc[(df_base_merge['Distribución analítica'].isna()) & 
            (df_base_merge['Número'].str.startswith('BNK')) &
                (df_base_merge['Cuenta Origen'].isin(['530515001 COMISIONES','530505002 GRAVAMEN CUATRO POR MIL', '530505001 CUOTA DE MANEJO']))
            , 'Distribución analítica'
            ] = '7'


df_base_merge.loc[(df_base_merge['Distribución analítica'].isna()) & 
            (df_base_merge['Número'].str.startswith('BNK')) &
                (df_base_merge['Cuenta Origen'].isin(['539595001 AJUSTE A MILES']))
            , 'Distribución analítica'
            ] = '6' 
df_base_merge.loc[(df_base_merge['Distribución analítica'].isna()) & 
            (df_base_merge['Número'].str.startswith('STJ')) 
            , 'Distribución analítica'
            ] = '6'

df_cc['cc'] = df_cc['cc'].astype(str)

df_base_merge = df_base_merge.merge(df_cc[['cc','Nombre Cencosto', 'ADM/VTAS','Origen' ]],
                                    left_on='Distribución analítica', right_on='cc', how='left').drop(columns='cc')
df_concepto['CC'] = df_concepto['CC'].str.upper().str.strip()

df_base_merge['Nombre Cencosto'] = df_base_merge['Nombre Cencosto'].str.upper().str.strip()

df_base_merge = df_base_merge.merge(df_concepto, left_on=['Cuenta','Nombre Cencosto' ], right_on=['Cuenta','CC'], how='left')

max_date = df_base_merge['Fecha'].max()
min_date = df_base_merge['Fecha'].min()
min_date.strftime('%d-%m-%Y')
ruta_base = ruta_contabilidad / 'base' / f'base_{min_date.strftime('%d-%m-%Y')}_{max_date.strftime('%d-%m-%Y')}.csv'
df_base_merge.to_csv(ruta_base, sep=";", index=False, encoding='utf-8', decimal=',')

centros_no_re = df_base_merge[(df_base_merge['Nombre Cencosto'].isna())&
            (~df_base_merge['Distribución analítica'].isna())
            ][['Distribución analítica ori','Distribución analítica', 'Nombre Cencosto' ]].drop_duplicates()
# Centros de costo mal clasificados
cc_corregir = df_base_merge[df_base_merge['Distribución analítica ori'].fillna('').str.count(':')>1]

# Genera el archivo de los casos sin centro de costos
sin_cc = df_base_merge[df_base_merge['Distribución analítica'].isna()]
sin_cc.to_excel(ruta_contabilidad / 'sin_cc.xlsx', index=False)

# Genera el archivo con los errores
with pd.ExcelWriter(ruta_contabilidad / 'correciones.xlsx', engine='openpyxl') as writer:
    sin_cc.to_excel(writer, index=False, sheet_name='Sin CC')
    cc_corregir.to_excel(writer, index=False, sheet_name='Corregir CC')
    centros_no_re.to_excel(writer, index=False, sheet_name='CC_no_registrados')

df_base_consol =  rc.consolidar_carpeta(extension='csv', encoding='utf-8', sep=';', decimal=',', ruta_carpeta= ruta_contabilidad / 'base')
# pd.read_csv(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\contabilidad\base\base_ene_jun_2025.csv",encoding='utf-8', sep=';')
df_base_consol.to_csv(ruta_contabilidad / 'base_consolidada.csv', encoding='utf-8', sep=';', decimal=',', index=False)

# return df_base_consol

# return df_base_consol

Buscando archivos con extensión '.xlsx' en: C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\contabilidad\odoo
  - Archivo 'Apunte contable (account.move.line) (3).xlsx' leído correctamente.
  - Archivo 'Apunte contable (account.move.line) (4).xlsx' leído correctamente.
  - Archivo 'Apunte contable (account.move.line) (5).xlsx' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!
Buscando archivos con extensión '.csv' en: C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\contabilidad\base


c:\Users\Dataa\Desktop\repo_analitica\clase_reportes.py:85: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_completa, sep=sep, encoding=encoding, decimal=decimal)


  - Archivo 'base_01-07-2025_31-07-2025.csv' leído correctamente.


c:\Users\Dataa\Desktop\repo_analitica\clase_reportes.py:85: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_completa, sep=sep, encoding=encoding, decimal=decimal)


  - Archivo 'base_01-08-2025_31-08-2025.csv' leído correctamente.


c:\Users\Dataa\Desktop\repo_analitica\clase_reportes.py:85: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_completa, sep=sep, encoding=encoding, decimal=decimal)


  - Archivo 'base_01-09-2025_30-09-2025.csv' leído correctamente.
  - Archivo 'base_01-10-2025_09-10-2025.csv' leído correctamente.


c:\Users\Dataa\Desktop\repo_analitica\clase_reportes.py:85: DtypeWarning: Columns (27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_completa, sep=sep, encoding=encoding, decimal=decimal)


  - Archivo 'base_ene_jun_2025.csv' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!


In [13]:
df_base_merge[df_base_merge['Número']=='FDS1858']

,Fecha,Diario,Número,Cuenta Origen,Creado el,NIT,Contacto,Referencia,Etiqueta,Importe en divisa,...,Nivel,Niveles,Distribución analítica,Nombre Cencosto,ADM/VTAS,Origen,Nombre cuenta concepto,Concepto,CC,Unnamed: 4
31776,2025-07-01,Documento Soporte,FDS1858,523560001 Publicidad Propaganda y Promocion,2025-07-10 13:22:47,901266653,META PLATFORMS IRELAND LIMITED,FDS1858,[PUBLICIDAD PROPAGANDA Y PROMOCION] PUBLICIDAD...,60.0,...,Gastos operacionales,Servicios,4,MARKETING,Gastos de venta y logistica,Cial,Publicidad Propaganda y Promocion,Gastos de Publicidad,MARKETING,NaN
31777,2025-07-01,Documento Soporte,FDS1858,523560010 Publicidad Propaganda y Promocion de...,2025-07-10 13:26:59,901266653,META PLATFORMS IRELAND LIMITED,FDS1858,[PUBLICIDAD PROPAGANDA EXTERIOR] SERVICIO PUBL...,13.0,...,Gastos operacionales,Servicios,4,MARKETING,Gastos de venta y logistica,Cial,NaN,NaN,NaN,NaN
31778,2025-07-01,Documento Soporte,FDS1858,523560010 Publicidad Propaganda y Promocion de...,2025-07-10 13:26:59,901266653,META PLATFORMS IRELAND LIMITED,FDS1858,[PUBLICIDAD PROPAGANDA EXTERIOR] SERVICIO PUBL...,446.0,...,Gastos operacionales,Servicios,4,MARKETING,Gastos de venta y logistica,Cial,NaN,NaN,NaN,NaN
